**1. Import Libraries and Mount to Drive**

In [ ]:
!pip install evaluate -q

# import libraries
import zipfile
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from datasets import load_dataset
import re
import evaluate
from tqdm import tqdm
import random
import math
import json
import numpy as np
from datetime import datetime
import pandas as pd
import json
import numpy as np
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00


**2. Load Model Qwen2.5-3B-Instruct (Base)**


In [ ]:
# base model name from hugging face
model_name = "Qwen/Qwen2.5-3B-Instruct"

# loads the tokenizer from the base model
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# add padding token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# loads the base model
print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto"
)
model.eval()

print("Model loaded and ready!")

Loading tokenizer...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded and ready!


**3. Load compute_overall_accuracy**

In [ ]:
# official accuracy computation function from medcalc-bench (table_stats.py)
def compute_overall_accuracy(output_path, model_name, prompt_style):
    category_accuracy = {}

    with open(f"outputs/{output_path}") as file:
        for line in file:
            data = json.loads(line)

            category = data["Category"]

            if category not in category_accuracy:
                category_accuracy[category] = []

            if data["Result"] == "Correct":
                category_accuracy[category].append(1)
            else:
                category_accuracy[category].append(0)

    # compute average and standard deviation for each category
    category_stats = {}
    all_results = []

    for cat, results in category_accuracy.items():
        results_array = np.array(results)
        category_mean = np.mean(results_array)
        category_std = round(np.sqrt(category_mean * (1-category_mean) / len(results_array)), 2)
        category_stats[cat] = {
            "average": round(category_mean * 100, 2),
            "std": category_std
        }
        all_results.extend(results)

    # compute overall average and standard deviation
    all_results_array = np.array(all_results)
    overall_average = np.mean(all_results_array)
    overall_std =  round(np.sqrt(overall_average * (1-overall_average) / 1047), 2)

    category_stats["overall"] = {
        "average": round(overall_average * 100, 2),
        "std": overall_std
    }

    if not os.path.exists("results"):
        os.makedirs("results")

    if "/" in model_name:
        model_name = model_name.split('/')[1]

    with open(f"results/results_{model_name}_{prompt_style}.json", "w") as file:
        json.dump(category_stats, file, indent=4)

    return category_stats

**4. Perform Evaluation (MedCalc-Bench-v1.0)**

In [ ]:
# set device to gpu if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# set seed for reproducibility
torch.manual_seed(42)

# official zero shot cot prompt from medcalc-bench
def zero_shot(note, question):
    system_msg = 'You are a helpful assistant for calculating a score for a given patient note. Please think step-by-step to solve the question and then generate the required score. Your output should only contain a JSON dict formatted as {"step_by_step_thinking": str(your_step_by_step_thinking_procress_to_solve_the_question), "answer": str(short_and_direct_answer_of_the_question)}.'
    user_temp = f'Here is the patient note:\n{note}\n\nHere is the task:\n{question}\n\nPlease directly output the JSON dict formatted as {{"step_by_step_thinking": str(your_step_by_step_thinking_procress_to_solve_the_question), "answer": str(short_and_direct_answer_of_the_question)}}:'
    return system_msg, user_temp

# official extract answer function from medcalc-bench
def extract_answer(answer, calid):
    calid = int(calid)
    extracted_answer = re.findall(r'[Aa]nswer":\s*(.*?)\}', answer)

    # tries to also grab the reasoning if it exists
    matches = re.findall(r'"step_by_step_thinking":\s*"([^"]+)"\s*,\s*"[Aa]nswer"', answer)
    if matches:
        explanation = matches[-1]
    else:
        explanation = "No Explanation"

    # checks if an answer was found
    if len(extracted_answer) == 0:
        extracted_answer = "Not Found"
    else:
        extracted_answer = extracted_answer[-1].strip().strip('"')
        if extracted_answer in [
            "str(short_and_direct_answer_of_the_question)",
            "str(value which is the answer to the question)",
            "X.XX"
        ]:
            extracted_answer = "Not Found"

    if calid in [13, 68]:
        match = re.search(r"^(0?[1-9]|1[0-2])\/(0?[1-9]|[12][0-9]|3[01])\/(\d{4})", extracted_answer)
        if match:
            month = int(match.group(1))
            day = int(match.group(2))
            year = match.group(3)
            answer = f"{month:02}/{day:02}/{year}"
        else:
            answer = "N/A"

    elif calid in [69]:
        extracted_answer = extracted_answer.replace("[", "(").replace("]", ")").replace("'", "").replace('"', "")
        match = re.search(r"\(?[\"\']?(\d+)\s*(weeks?)?[\"\']?,?\s*[\"\']?(\d+)\s*(days?)?[\"\']?\s*\)?", extracted_answer)
        if match:
            weeks = match.group(1)
            days = match.group(3)
            answer = f"({weeks}, {days})"
        else:
            answer = "N/A"

    elif calid in [4, 15, 16, 17, 18, 20, 21, 25, 27, 28, 29, 32, 33, 36, 43, 45, 48, 51]:
        match = re.search(r"(\d+) out of", extracted_answer)
        if match:
            answer = match.group(1)
        else:
            match = re.search(r"-?\d+(, ?-?\d+)+", extracted_answer)
            if match:
                answer = str(len(match.group(0).split(",")))
            else:
                match = re.findall(r"(-?\d+(\.\d+)?)", extracted_answer)
                if len(match) > 0:
                    answer = match[-1][0]
                else:
                    answer = "N/A"

    elif calid in [2, 3, 5, 6, 7, 8, 9, 10, 11, 19, 22, 23, 24, 26, 30, 31, 38, 39, 40, 44, 46, 49, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67]:
        match = re.search(r"str\((.*)\)", extracted_answer)
        if match:
            expression = match.group(1).replace("^", "**").replace("is odd", "% 2 == 1").replace("is even", "% 2 == 0").replace("sqrt", "math.sqrt").replace(".math", "").replace("weight", "").replace("height", "").replace("mg/dl", "").replace("g/dl", "").replace("mmol/L", "").replace("kg", "").replace("g", "").replace("mEq/L", "")
            expression = expression.split('#')[0]
            if expression.count('(') > expression.count(')'):
                expression += ')' * (expression.count('(') - expression.count(')'))
            elif expression.count(')') > expression.count('('):
                expression = '(' * (expression.count(')') - expression.count('(')) + expression
            try:
                answer = eval(expression, {"__builtins__": None}, {"min": min, "pow": pow, "round": round, "abs": abs, "int": int, "float": float, "math": math, "np": np, "numpy": np})
            except:
                answer = "N/A"
        else:
            match = re.search(r"(-?\d+(\.\d+)?)\s*mL/min/1.73", extracted_answer)
            if match:
                answer = eval(match.group(1))
            else:
                match = re.findall(r"(-?\d+(\.\d+)?)\%", extracted_answer)
                if len(match) > 0:
                    answer = eval(match[-1][0]) / 100
                else:
                    match = re.findall(r"(-?\d+(\.\d+)?)", extracted_answer)
                    if len(match) > 0:
                        answer = eval(match[-1][0])
                    else:
                        answer = "N/A"
        if answer != "N/A":
            answer = str(answer)
    else:
        answer = "N/A"

    return answer, explanation

# official correctness check from medcalc-bench
def check_correctness(answer, ground_truth, calid, upper_limit, lower_limit):
    calid = int(calid)
    try:
        if calid in [13, 68]:
            if datetime.strptime(answer, "%m/%d/%Y").strftime("%-m/%-d/%Y") == datetime.strptime(ground_truth, "%m/%d/%Y").strftime("%-m/%-d/%Y"):
                correctness = 1
            else:
                correctness = 0

        elif calid in [69]:
            match = re.search(r"\(?[\"\']?(\d+)\s*(weeks?)?[\"\']?,?\s*[\"\']?(\d+)\s*(days?)?[\"\']?\s*\)?", ground_truth)
            ground_truth = f"({match.group(1)}, {match.group(3)})"
            match = re.search(r"\(?[\"\']?(\d+)\s*(weeks?)?[\"\']?,?\s*[\"\']?(\d+)\s*(days?)?[\"\']?\s*\)?", answer)
            if match:
                weeks = match.group(1)
                days = match.group(3)
                answer = f"({weeks}, {days})"
                correctness = 1 if eval(answer) == eval(ground_truth) else 0
            else:
                correctness = 0

        elif calid in [4, 15, 16, 17, 18, 20, 21, 25, 27, 28, 29, 32, 33, 36, 43, 45, 48, 51]:
            answer = round(eval(answer))
            correctness = 1 if answer == eval(ground_truth) else 0

        elif calid in [2, 3, 5, 6, 7, 8, 9, 10, 11, 19, 22, 23, 24, 26, 30, 31, 38, 39, 40, 44, 46, 49, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67]:
            answer = eval(answer)
            correctness = 1 if eval(lower_limit) <= answer <= eval(upper_limit) else 0

        else:
            return None

    except Exception:
        return None

    return correctness

# load the official test set
df = pd.read_csv("test_data.csv")
test_set = df

print(f"Running evaluation on all {len(test_set)} questions...")

correct_count = 0
wrong_count = 0
skipped_count = 0
total = len(test_set)

category_stats = {}

output_path = f"{model_name.replace('/', '_')}_zero_shot.jsonl"

if not os.path.exists("outputs"):
    os.makedirs("outputs")

# set eos tokens based on model family
if "llama" in model_name.lower():
    eos_token_ids = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")]
elif "qwen" in model_name.lower():
    eos_token_ids = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")]
else:
    eos_token_ids = [tokenizer.eos_token_id]

# official loop from medcalc-bench
for index in tqdm(range(len(test_set))):
    example = test_set.iloc[index]

    system, user = zero_shot(example["Patient Note"], example["Question"])

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
    input_length = inputs["input_ids"].shape[1]

    # official-style generation limit using the model's actual context length
    model_context_length = getattr(model.config, "max_position_embeddings", tokenizer.model_max_length)

    # fallback if tokenizer has an unrealistic huge placeholder value
    if model_context_length is None or model_context_length > 1_000_000:
        model_context_length = tokenizer.model_max_length

    max_new_tokens = min(4096, max(1, model_context_length - input_length))

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=eos_token_ids,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    response = re.sub(r"\s+", " ", response)

    try:
        predicted, explanation = extract_answer(response, example['Calculator ID'])
    except Exception as e:
        predicted = "N/A"
        explanation = str(e)

    if predicted not in ["N/A", "Not Found", None]:
        try:
            correctness = check_correctness(
                answer=predicted,
                ground_truth=example['Ground Truth Answer'],
                calid=example['Calculator ID'],
                upper_limit=example['Upper Limit'],
                lower_limit=example['Lower Limit']
            )
        except Exception:
            correctness = None
    else:
        correctness = None

    if correctness is None:
        skipped_count += 1
        result_str = "Skipped"
    elif correctness == 1:
        correct_count += 1
        result_str = "Correct"
    else:
        wrong_count += 1
        result_str = "Wrong"

    cat = example.get('Category', 'Unknown')
    if cat not in category_stats:
        category_stats[cat] = {'correct': 0, 'wrong': 0, 'skipped': 0}
    if correctness is None:
        category_stats[cat]['skipped'] += 1
    elif correctness == 1:
        category_stats[cat]['correct'] += 1
    else:
        category_stats[cat]['wrong'] += 1

    print(f"\n{'-' * 50}")
    print(f"Question {index+1} of {total}")
    print(f"Calculator:       {example['Calculator Name']}")
    print(f"Output type:      {example['Output Type']}")
    print(f"Ground truth:     {example['Ground Truth Answer']}")
    print(f"Acceptable range: {example['Lower Limit']} - {example['Upper Limit']}")
    print(f"Model response:   {response}")
    print(f"Extracted answer: {predicted}")
    print(f"Result:           {result_str}")
    print(f"{'-' * 50}")

    with open(f"outputs/{output_path}", "a") as f:
        f.write(json.dumps({
            "Row Number": int(example["Row Number"]),
            "Calculator Name": example['Calculator Name'],
            "Calculator ID": str(example['Calculator ID']),
            "Category": example['Category'],
            "Note ID": str(example['Note ID']),
            "Patient Note": example['Patient Note'],
            "Question": example['Question'],
            "LLM Answer": predicted,
            "LLM Explanation": explanation,
            "Ground Truth Answer": example['Ground Truth Answer'],
            "Ground Truth Explanation": example['Ground Truth Explanation'],
            "Result": "Correct" if correctness == 1 else "Incorrect"
        }) + "\n")

# calculates the final stats
answered = correct_count + wrong_count
coverage = (answered / total * 100)
accuracy_all = (correct_count / total * 100)

print("\n" + "=" * 50)
print("EVALUATION COMPLETE")
print(f"Total questions:          {total}")
print(f"Correct answers:          {correct_count}")
print(f"Wrong answers:            {wrong_count}")
print(f"Skipped/Invalid:          {skipped_count}")
print(f"Coverage:                 {coverage:.2f}%")
print(f"Accuracy (all {total}):   {accuracy_all:.2f}%")
print("=" * 50)

# official accuracy checker
compute_overall_accuracy(output_path, model_name, "zero_shot")

Running evaluation on all 1047 questions...


  0%|          | 1/1047 [00:06<1:46:50,  6.13s/it]


--------------------------------------------------
Question 1 of 1047
Calculator:       Creatinine Clearance (Cockcroft-Gault Equation)
Output type:      decimal
Ground truth:     25.238
Acceptable range: 23.9761 - 26.4999
Model response:   { "step_by_step_thinking": "First, I will calculate the patient's Body Mass Index (BMI) using the provided information. Then, I will determine if the patient is overweight or obese based on their BMI. After that, I will decide whether to use the adjusted body weight or the actual body weight for the Cockroft-Gault equation calculation. Finally, I will apply the Cockroft-Gault formula to find the Creatinine Clearance.", "answer": "12.8" }
Extracted answer: 12.8
Result:           Wrong
--------------------------------------------------


  0%|          | 2/1047 [00:14<2:08:27,  7.38s/it]


--------------------------------------------------
Question 2 of 1047
Calculator:       Creatinine Clearance (Cockcroft-Gault Equation)
Output type:      decimal
Ground truth:     38
Acceptable range: 36.1 - 39.9
Model response:   { "step_by_step_thinking": "First, we need to calculate the patient's Body Mass Index (BMI) to determine if they are overweight or obese. Then, we will use the Cockroft-Gault equation to calculate the Creatinine Clearance. Since the patient's BMI is not provided, we will assume normal weight for the purpose of this calculation. We will use the minimum of the ideal body weight and actual body weight as the adjusted body weight. The ideal body weight can be calculated using the formula: Ideal Weight = (height in meters * 703) - 453.6. Finally, we will apply the Cockroft-Gault equation: Creatinine Clearance (CC) = (140 - age) * weight (in kg) / (0.817 * serum creatinine (in mg/dL)).", "answer": "12.4" }
Extracted answer: 12.4
Result:           Wrong
-----------

...

100%|█████████▉| 1046/1047 [2:10:59<00:05,  5.56s/it]


--------------------------------------------------
Question 1046 of 1047
Calculator:       Estimated Gestational Age
Output type:      integer
Ground truth:     ('34 weeks', '6 days')
Acceptable range: ('34 weeks', '6 days') - ('34 weeks', '6 days')
Model response:   { "step_by_step_thinking": "To estimate the gestational age from the last menstrual period (LMP) to the current date, we need to calculate the difference in days between these two dates. The LMP is on July 18, 2013, and the current date is March 19, 2014. We can break this down into months and days to simplify the calculation.", "answer": "(26 weeks, 1 day)" }
Extracted answer: (26, 1)
Result:           Wrong
--------------------------------------------------


100%|██████████| 1047/1047 [2:11:03<00:00,  7.51s/it]


--------------------------------------------------
Question 1047 of 1047
Calculator:       Estimated Gestational Age
Output type:      integer
Ground truth:     ('14 weeks', '2 days')
Acceptable range: ('14 weeks', '2 days') - ('14 weeks', '2 days')
Model response:   { "step_by_step_thinking": "To estimate the gestational age from the last menstrual period (LMP) to the current date, we need to calculate the difference in days between these two dates. The LMP is on October 16, 2007, and the current date is January 24, 2008. We can break this down into months and days to simplify the calculation.", "answer": "(14 weeks, 8 days)" }
Extracted answer: (14, 8)
Result:           Wrong
--------------------------------------------------

EVALUATION COMPLETE
Total questions:          1047
Correct answers:          109
Wrong answers:            812
Skipped/Invalid:          126
Coverage:                 87.97%
Accuracy (all 1047):   10.41%


{'lab': {'average': np.float64(10.4), 'std': np.float64(0.02)},
 'risk': {'average': np.float64(12.5), 'std': np.float64(0.02)},
 'physical': {'average': np.float64(10.83), 'std': np.float64(0.02)},
 'severity': {'average': np.float64(7.5), 'std': np.float64(0.03)},
 'diagnosis': {'average': np.float64(18.33), 'std': np.float64(0.05)},
 'date': {'average': np.float64(1.67), 'std': np.float64(0.02)},
 'dosage': {'average': np.float64(2.5), 'std': np.float64(0.02)},
 'overall': {'average': np.float64(10.41), 'std': np.float64(0.01)}}